In [2]:
:clear

In [3]:
:dep pecos = { version = "0.1.1", path = "../crates/pecos" }

In [4]:
// Bell State Example using SparseStab simulator
use pecos::prelude::*;

// Create a 2-qubit stabilizer simulator
let mut sim = StdSparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sim.h(0)      // Apply Hadamard to qubit 0: |00⟩ -> (|00⟩ + |10⟩)/√2
   .cx(0, 1); // Apply CNOT: (|00⟩ + |10⟩)/√2 -> (|00⟩ + |11⟩)/√2

// Measure both qubits
let result0: MeasurementResult = sim.mz(0);
let result1: MeasurementResult = sim.mz(1);

println!("Bell State Measurement Results:");
println!("  Qubit 0: {} (deterministic: {})", 
         if result0.outcome { "1" } else { "0" },
         result0.is_deterministic);
println!("  Qubit 1: {} (deterministic: {})", 
         if result1.outcome { "1" } else { "0" },
         result1.is_deterministic);

// Verify Bell state correlation: both qubits should always measure the same
assert_eq!(result0.outcome, result1.outcome, 
           "Bell state measurements must be correlated!");
println!("\nSuccess! Measurements are correlated as expected for a Bell state.");

Bell State Measurement Results:
  Qubit 0: 1 (deterministic: false)
  Qubit 1: 1 (deterministic: true)

Success! Measurements are correlated as expected for a Bell state.


In [5]:
// Symbolic Bell State Example using SymbolicSparseStab
// This simulator tracks measurement dependencies instead of collapsing to concrete outcomes

let mut sym_sim = StdSymbolicSparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sym_sim.h(0).cx(0, 1);

// Measure both qubits
let r0: SymbolicMeasurementResult = sym_sim.mz(0);
let r1: SymbolicMeasurementResult = sym_sim.mz(1);

println!("Symbolic Bell State Measurement Results:");
println!("  Measurement 0: {} (deterministic: {})", r0, r0.is_deterministic);
println!("  Measurement 1: {} (deterministic: {})", r1, r1.is_deterministic);

// The first measurement is non-deterministic (creates measurement index 0)
// The second measurement is deterministic and depends on measurement 0
println!("\nAnalysis:");
if !r0.is_deterministic {
    println!("  - Qubit 0 measurement was non-deterministic (random outcome)");
}
if r1.is_deterministic && r0.outcome == r1.outcome {
    println!("  - Qubit 1 measurement is deterministic and depends on measurement 0");
    println!("  - This shows the Bell state correlation: both qubits always measure the same!");
}

Symbolic Bell State Measurement Results:
  Measurement 0: m0=? (deterministic: false)
  Measurement 1: m1^m0=0 (deterministic: true)

Analysis:
  - Qubit 0 measurement was non-deterministic (random outcome)
  - Qubit 1 measurement is deterministic and depends on measurement 0
  - This shows the Bell state correlation: both qubits always measure the same!


()

In [6]:
// 3-qubit Repetition Code in Logical |+_L⟩ with Syndrome Measurements
//
// Qubits:
//   q0, q1, q2: Data qubits 
//   q3: Ancilla for Z0Z1 check
//   q4: Ancilla for Z1Z2 check
//
// Logical states:
//   |0_L⟩ = |000⟩
//   |1_L⟩ = |111⟩
//   |+_L⟩ = (|000⟩ + |111⟩)/√2
//
// Encoding circuit for |+_L⟩:
//   - Start with |+⟩|00⟩ (H on q0)
//   - CX(0,1), CX(0,2) to spread → (|000⟩ + |111⟩)/√2
//
// Syndrome extraction:
//   - Measure Z0Z1 using ancilla q3
//   - Measure Z1Z2 using ancilla q4
//
// Then measure data qubits in Z basis

let mut sim = StdSymbolicSparseStab::new(5);

println!("=== 3-Qubit Repetition Code: Logical |+_L⟩ ===\n");

// Encode logical |+_L⟩ = (|000⟩ + |111⟩)/√2
sim.h(0);           // |+⟩|00⟩
sim.cx(0, 1);       // (|00⟩ + |11⟩)|0⟩
sim.cx(0, 2);       // |000⟩ + |111⟩
println!("After encoding |+_L⟩ = (|000⟩ + |111⟩)/√2:");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z0Z1 via ancilla q3
// Circuit: H(q3), CX(q0,q3), CX(q1,q3), H(q3), Measure(q3)
sim.h(3);
sim.cx(0, 3);
sim.cx(1, 3);
sim.h(3);
println!("After Z0Z1 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s0: SymbolicMeasurementResult = sim.mz(3);
println!("Syndrome S0 (Z0Z1) = {}, det={}", s0, s0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z1Z2 via ancilla q4
sim.h(4);
sim.cx(1, 4);
sim.cx(2, 4);
sim.h(4);
println!("After Z1Z2 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s1: SymbolicMeasurementResult = sim.mz(4);
println!("Syndrome S1 (Z1Z2) = {}, det={}", s1, s1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure data qubits in Z basis
let d0: SymbolicMeasurementResult = sim.mz(0);
println!("D0 (q0) = {}, det={}", d0, d0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let d1: SymbolicMeasurementResult = sim.mz(1);
println!("D1 (q1) = {}, det={}", d1, d1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let d2: SymbolicMeasurementResult = sim.mz(2);
println!("D2 (q2) = {}, det={}", d2, d2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

println!("\n=== Measurement History ===");
println!("Total measurements: {}", sim.measurement_history().len());
println!("Deterministic: {}", sim.measurement_history().deterministic().len());
println!("Non-deterministic: {}", sim.measurement_history().nondeterministic().len());

// Use the new formatting convenience methods
println!("\nAll measurements:       {}", sim.measurement_history());
println!("Deterministic only:     {}", sim.measurement_history().format_deterministic());
println!("Non-deterministic only: {}", sim.measurement_history().format_nondeterministic());

// Show Debug format for one result (wrap in block to avoid lifetime issue)
println!("\nDebug format for first measurement:");
{
    let first = &sim.measurement_history()[0];
    println!("  {:?}", first);
}

println!("\n=== Summary ===");
println!("Measurement indices: S0=0, S1=1, D0=2, D1=3, D2=4");
println!("\nSyndromes should be deterministic {{}} (no errors in code space).");
println!("Data qubits: D0 is random, D1 and D2 are determined by D0 and syndromes.");
println!("\nDetectors: {{S0=0, S1=0, D1^D0=0, D2^D0=0}}")

=== 3-Qubit Repetition Code: Logical |+_L⟩ ===

After encoding |+_L⟩ = (|000⟩ + |111⟩)/√2:
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

After Z0Z1 syndrome circuit (before measurement):
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

Syndrome S0 (Z0Z1) = m0=0, det=true
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

After Z1Z2 syndrome circuit (before measurement):
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

Syndrome S1 (Z1Z2) = m1=0, det=true
Stabilizers:
{} ^ 0 XXXII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

D0 (q0) = m2=?, det=false
Stabilizers:
{2} ^ 0 ZIIII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

D1 (q1) = m3^m2=0, det=true
Stabilizers:
{2} ^ 0 ZIIII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ

D2 (q2) = m4^m2=0, det=true
Stabilizers:
{2} ^ 0 ZIIII
{} ^ 0 ZZIII
{} ^ 0 ZIZII
{} ^ 0 IIIZI
{} ^ 0 IIIIZ


=== Measurement His

()

In [7]:
println!("Measurement history: {}", sim.measurement_history());

Measurement history: [m0=0, m1=0, m2=?, m3^m2=0, m4^m2=0]


In [8]:
// More complex example: 3-qubit circuit with interesting measurement dependencies
// 
// Circuit:
//   q0: --H--@-------M0
//            |
//   q1: -----X--H--@-M1
//                  |
//   q2: -----------X-M2
//
// This creates a chain where:
// - q0 and q1 become entangled (Bell pair)
// - Then q1 and q2 become entangled
// - Measuring in this order should show interesting dependencies

let mut sim = StdSymbolicSparseStab::new(3);

// Create the entanglement chain
sim.h(0);        // q0 in superposition
sim.cx(0, 1);    // Entangle q0-q1
sim.h(1);        // q1 in superposition (relative to q0)
sim.cx(1, 2);    // Entangle q1-q2

// Measure all qubits
let m0: SymbolicMeasurementResult = sim.mz(0);
let m1: SymbolicMeasurementResult = sim.mz(1);
let m2: SymbolicMeasurementResult = sim.mz(2);

println!("3-Qubit Chain Measurement Results:");
println!("  M0 (qubit 0): {} (deterministic: {})", m0, m0.is_deterministic);
println!("  M1 (qubit 1): {} (deterministic: {})", m1, m1.is_deterministic);
println!("  M2 (qubit 2): {} (deterministic: {})", m2, m2.is_deterministic);

println!("\nInterpretation:");
println!("  - M0 outcome is random: {}", m0);
println!("  - M1 outcome is random: {}", m1);
println!("  - M2 outcome depends on: {}", m2);

// Show what XOR means
if m2.outcome.len() == 2 {
    println!("\n  M2 = {{0, 1}} means: M2_outcome = M0_outcome XOR M1_outcome");
}

3-Qubit Chain Measurement Results:
  M0 (qubit 0): m0=? (deterministic: false)
  M1 (qubit 1): m1=? (deterministic: false)
  M2 (qubit 2): m2^m1=0 (deterministic: true)

Interpretation:
  - M0 outcome is random: m0=?
  - M1 outcome is random: m1=?
  - M2 outcome depends on: m2^m1=0


()

In [9]:
// Trace through the 3-qubit circuit step by step with tableau display
// Note: The tableau format is "{measurement_indices} ^ flip PauliString"
//   - {} ^ 0: identity sign, no flip (deterministic 0)
//   - {} ^ 1: identity sign, flipped (deterministic 1)
//   - {0} ^ 0: depends on measurement 0, no flip
//   - {0,1} ^ 1: XOR of measurements 0,1, flipped
//
// Measurement results use Display format: m{index}^m{dep1}^m{dep2}^...={flip}
//   - m0=0: measurement 0 is deterministic 0
//   - m1^m0=0: measurement 1 equals measurement 0
//   - m2^m1^m0=1: measurement 2 equals m1 XOR m0 XOR 1

let mut sim = StdSymbolicSparseStab::new(3);

println!("Initial state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(0);
println!("After H(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(0, 1);
println!("After CX(0,1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(1);
println!("After H(1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(1, 2);
println!("After CX(1,2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After measuring qubit 0 ({}, det={}):", m0, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After measuring qubit 1 ({}, det={}):", m1, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After measuring qubit 2 ({}, det={}):", m2, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

Initial state:
Stabilizers:
{} ^ 0 ZII
{} ^ 0 IZI
{} ^ 0 IIZ

After H(0):
Stabilizers:
{} ^ 0 XII
{} ^ 0 IZI
{} ^ 0 IIZ

After CX(0,1):
Stabilizers:
{} ^ 0 XXI
{} ^ 0 ZZI
{} ^ 0 IIZ

After H(1):
Stabilizers:
{} ^ 0 XZI
{} ^ 0 ZXI
{} ^ 0 IIZ

After CX(1,2):
Stabilizers:
{} ^ 0 XZI
{} ^ 0 ZXX
{} ^ 0 IZZ

After measuring qubit 0 (m0=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZXX
{} ^ 0 IZZ

After measuring qubit 1 (m1=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{1} ^ 0 IZI
{} ^ 0 IZZ

After measuring qubit 2 (m2^m1=0, det=true):
Stabilizers:
{0} ^ 0 ZII
{1} ^ 0 IZI
{} ^ 0 IZZ



In [10]:
// Example where M2 depends on BOTH M0 and M1 (XOR)
//
// Circuit: Create a 3-qubit GHZ-like state, then do local rotations
//
//   q0: --H--@--------M0
//            |
//   q1: -----X--@-----M1
//               |
//   q2: --------X--H--M2
//
// The key insight: after CX gates, we have correlations.
// The H on q2 at the end converts Z2 to X2, which should
// create an XOR dependency when measured in Z basis.

let mut sim = StdSymbolicSparseStab::new(3);

println!("=== Circuit where M2 = M0 XOR M1 ===\n");

sim.h(0);
sim.cx(0, 1);
sim.cx(1, 2);
println!("After H(0), CX(0,1), CX(1,2) - GHZ state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(2);
println!("After H(2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After M0 ({}, det={}):", m0, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After M1 ({}, det={}):", m1, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After M2 ({}, det={}):", m2, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

println!("Summary:");
println!("  M0 = {}", m0);
println!("  M1 = {}", m1);
println!("  M2 = {}", m2);
if m2.outcome.len() == 2 && m2.outcome.contains(&0) && m2.outcome.contains(&1) {
    println!("\n  m2^m1^m0=0 means: M2_outcome = M0_outcome XOR M1_outcome");
}

=== Circuit where M2 = M0 XOR M1 ===

After H(0), CX(0,1), CX(1,2) - GHZ state:
Stabilizers:
{} ^ 0 XXX
{} ^ 0 ZZI
{} ^ 0 IZZ

After H(2):
Stabilizers:
{} ^ 0 XXZ
{} ^ 0 ZZI
{} ^ 0 IZX

After M0 (m0=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZZI
{} ^ 0 IZX

After M1 (m1^m0=0, det=true):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZZI
{} ^ 0 IZX

After M2 (m2=?, det=false):
Stabilizers:
{0} ^ 0 ZII
{} ^ 0 ZZI
{2} ^ 0 IIZ

Summary:
  M0 = m0=?
  M1 = m1^m0=0
  M2 = m2=?


()

In [11]:
// Example demonstrating the flip flag from unitary operations
//
// The flip flag tracks phase changes from unitary gates:
// - X gate on |0⟩: Z stabilizer becomes -Z, so flip=1
// - Measurement of |1⟩ returns m0=1 (deterministic 1)

let mut sim = StdSymbolicSparseStab::new(1);

println!("=== Demonstrating the flip flag ===\n");

println!("Initial state |0⟩:");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Apply X gate to flip |0⟩ to |1⟩
sim.x(0);
println!("After X gate (now |1⟩):");
println!("Stabilizers:\n{}", sim.stab_tableau());
println!("Notice: {{}} ^ 1 Z means the stabilizer is -Z (flip=1)\n");

// Measure - should be deterministic 1
let m: SymbolicMeasurementResult = sim.mz(0);
println!("Measurement result: {}", m);
println!("  - Empty dependency set means no measurement dependencies");
println!("  - =1 means the result is flipped, giving outcome 1");

// Reset and show double-X cancellation
sim.reset();
sim.x(0);
sim.x(0);
println!("\nAfter X·X (identity):");
println!("Stabilizers:\n{}", sim.stab_tableau());
println!("Notice: {{}} ^ 0 Z means we're back to +Z (flip=0)")

=== Demonstrating the flip flag ===

Initial state |0⟩:
Stabilizers:
{} ^ 0 Z

After X gate (now |1⟩):
Stabilizers:
{} ^ 1 Z

Notice: {} ^ 1 Z means the stabilizer is -Z (flip=1)

Measurement result: m0=1
  - Empty dependency set means no measurement dependencies
  - =1 means the result is flipped, giving outcome 1

After X·X (identity):
Stabilizers:
{} ^ 0 Z

Notice: {} ^ 0 Z means we're back to +Z (flip=0)


()

In [12]:
// Example combining measurement dependencies with flip flag
//
// Circuit: X on qubit 0, then create Bell state
//   q0: --X--H--@--M0
//               |
//   q1: --------X--M1
//
// This shows how the flip propagates through the circuit

let mut sim = StdSymbolicSparseStab::new(2);

println!("=== Combining measurement dependencies with flip ===\n");

sim.x(0);
println!("After X(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(0);
println!("After H(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(0, 1);
println!("After CX(0,1) - Bell state with phase:");
println!("Stabilizers:\n{}", sim.stab_tableau());

let m0: SymbolicMeasurementResult = sim.mz(0);
let m1: SymbolicMeasurementResult = sim.mz(1);

println!("Measurement results:");
println!("  M0: {}", m0);
println!("  M1: {}", m1);
println!("\nBoth measurements depend on measurement index 0.");
println!("The flips indicate phase accumulated from the initial X gate.")

=== Combining measurement dependencies with flip ===

After X(0):
Stabilizers:
{} ^ 1 ZI
{} ^ 0 IZ

After H(0):
Stabilizers:
{} ^ 1 XI
{} ^ 0 IZ

After CX(0,1) - Bell state with phase:
Stabilizers:
{} ^ 1 XX
{} ^ 0 ZZ

Measurement results:
  M0: m0=?
  M1: m1^m0=0

Both measurements depend on measurement index 0.
The flips indicate phase accumulated from the initial X gate.


()